In [1]:
## Use meta-llama/Llama-3.2-11B-Vision-Instruct
## to create results for text based problems

from google.colab import userdata
from huggingface_hub import login
import json
import time
import requests
import torch
from transformers import AutoTokenizer, MllamaForCausalLM

In [2]:
## check if cuda is available and GPU

print("CUDA available:", torch.cuda.is_available())
print("Using GPU:", torch.cuda.get_device_name(0))

CUDA available: True
Using GPU: NVIDIA L4


In [3]:
## authenticate with huggingface

hf_token = userdata.get("hf_token")
login(token=hf_token)

In [4]:
# Choose a precision supported by the GPU
dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

In [5]:
model_name="Llama-VI"
model_id="meta-llama/Llama-3.2-11B-Vision-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id,clean_up_tokenization_spaces=False,)

# Only the language component is loaded as input is text
model = MllamaForCausalLM.from_pretrained(
    model_id,
    dtype=dtype,
    device_map="auto",
)


config.json:   0%|          | 0.00/5.07k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/89.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/395 [00:00<?, ?it/s]

[transformers] MllamaForCausalLM LOAD REPORT from: meta-llama/Llama-3.2-11B-Vision-Instruct
Key                                                                                             | Status     |  | 
------------------------------------------------------------------------------------------------+------------+--+-
vision_model.global_transformer.layers.{0, 1, 2, 3, 4, 5, 6, 7}.post_attention_layernorm.bias   | UNEXPECTED |  | 
vision_model.transformer.layers.{0...31}.mlp.fc2.weight                                         | UNEXPECTED |  | 
vision_model.transformer.layers.{0...31}.post_attention_layernorm.bias                          | UNEXPECTED |  | 
vision_model.transformer.layers.{0...31}.self_attn.v_proj.weight                                | UNEXPECTED |  | 
vision_model.transformer.layers.{0...31}.mlp.fc1.weight                                         | UNEXPECTED |  | 
vision_model.transformer.layers.{0...31}.self_attn.q_proj.weight                                | UNEXP

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [6]:
## import the prompts from the image_prompts.py

import sys

sys.path.append("/content/drive/MyDrive/colab_files/Problems/Text")
from text_prompts import prompts

print(prompts)

{'regular_system_prompt': 'You are a physics expert. Based on the question that is given solve the problem. The question has equations and symbols in latex, but your answer should not be in latex.', 'detail_system_prompt': 'You are a physics expert. Based on the question that is given solve the problem, but if there are any physical laws broken then mention it and also say if the problem can be solved or not. The question has equations and symbols in latex, but your answer should not be in latex.'}


In [7]:
print(prompts.keys())

dict_keys(['regular_system_prompt', 'detail_system_prompt'])


In [8]:
from pathlib import Path

text_folder = Path("/content/drive/MyDrive/colab_files/Problems/Text")
file_names = sorted(file.name for file in text_folder.glob("*.md"))
print("Total number of files:::",len(file_names))
print(file_names)

Total number of files::: 24
['Problem_1_Obvious_Q.md', 'Problem_1_Regular_Q.md', 'Problem_1_non_obvious_Q.md', 'Problem_2_Obvious_Q.md', 'Problem_2_Regular_Q.md', 'Problem_2_non_obvious_Q.md', 'Problem_3_Obvious_Q.md', 'Problem_3_Regular_Q.md', 'Problem_3_non_obvious_Q.md', 'Problem_4_Obvious_Q.md', 'Problem_4_Regular_Q.md', 'Problem_4_non_obvious_Q.md', 'Problem_5_Obvious_Q.md', 'Problem_5_Regular_Q.md', 'Problem_5_non_obvious_Q.md', 'Problem_6_Obvious_Q.md', 'Problem_6_Regular_Q.md', 'Problem_6_non_obvious_Q.md', 'Problem_7_Obvious_Q.md', 'Problem_7_Regular_Q.md', 'Problem_7_non_obvious_Q.md', 'Problem_8_Obvious_Q.md', 'Problem_8_Regular_Q.md', 'Problem_8_non_obvious_Q.md']


In [9]:
## to calculate the total time taken to generate the output

def synchronize_cuda():
    # GPU operations are asynchronous.
    # Wait for pending operations before taking timing boundaries.
    if torch.cuda.is_available():
        for device_index in range(torch.cuda.device_count()):
            torch.cuda.synchronize(device_index)

In [10]:
%%time

for keys in prompts:
  for file in file_names:

    text_folder = Path("/content/drive/MyDrive/colab_files/Problems/Text") ## read the files
    file_path = text_folder/ file

    if keys == "detail_system_prompt" and "regular" in file_path.stem.lower():  ## skip if keys have detail_system_prompt and
      continue     ## problems that have regular in their name. We dont need Regular problems to use detail prompts but run the rest

    problem_text = file_path.read_text(encoding="utf-8")

    messages = [
          {
              "role": "system",
              "content": prompts[keys],
            },
        {
              "role": "user",
              "content": problem_text,
          }
              ]

    # Format and tokenize the conversation
    inputs = tokenizer.apply_chat_template(
      messages,
      add_generation_prompt=True,
      tokenize=True,
      return_dict=True,
      return_tensors="pt",
          ).to(model.device)

    input_length = inputs["input_ids"].shape[-1]

    ## generate and measure time
    synchronize_cuda()
    start_time = time.perf_counter()

    # Generate the response
    with torch.inference_mode():
        outputs = model.generate(
          **inputs,
         max_new_tokens=512,
          do_sample=False,
          pad_token_id=tokenizer.eos_token_id,
     )

    synchronize_cuda()
    end_time = time.perf_counter()

    ## calculate time taken
    total_time_seconds = end_time - start_time

    # Decode only the newly generated tokens
    generated_tokens = outputs[0, input_length:]
    output_tokens = generated_tokens.numel()

    response = tokenizer.decode(
      generated_tokens,
      skip_special_tokens=True,
          )
    output_text=response
    output_filename= f"{model_name}_{file_path.stem}_{keys}" +".md"

    output_folder = Path("/content/drive/MyDrive/colab_files/Results/llama_VI_text")
    output_file_path = output_folder / output_filename
    output_file_path.write_text(output_text, encoding="utf-8")

    ## Build metadata

    metadata = {
    "requested_model": model_id,
    "problem_text": problem_text,
    "prompt_name": keys,
    "total_output_tokens": output_tokens,
    "total_time_taken": total_time_seconds
    }

    metadata_folder= Path("/content/drive/MyDrive/colab_files/Results/llama_VI_text/metadata")
    metadata_name= f"{model_name}_{file_path.stem}_{keys}" +".json"

    meta_data_file_path = metadata_folder / metadata_name
    meta_data_file_path.write_text(
                        json.dumps(metadata, indent=4, ensure_ascii=False),
                        encoding="utf-8",
                    )

    print(f"Completed:::{model_name}_{file_path.stem}_{keys}")

Completed:::Llama-VI_Problem_1_Obvious_Q_regular_system_prompt


KeyboardInterrupt: 